<a href="https://colab.research.google.com/github/bahaali273/COMP9170-Hospital-Readmission/blob/shaaz/MIMIC_(demo)_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## We are doing EDA on the [MIMIC-IV Clinical Database Demo](https://physionet.org/content/mimic-iv-demo/2.2/icu/#description) to check whether it fits our study before we proceed with requsting acces to the full data set.

### Mount Google Drive and Locate the Zip File
We will start by mounting your Google Drive to access the uploaded dataset.

In [1]:
from google.colab import drive
import os
import zipfile
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Define the zip path and extraction destination
zip_path = '/content/drive/My Drive/Colab Notebooks/mimic_iv/mimic-iv-clinical-database-demo-2.2.zip'
extract_path = '/content/mimic_demo'

print(f"Zip file exists: {os.path.exists(zip_path)}")

Mounted at /content/drive
Zip file exists: True


### Extract the Zip File

In [2]:
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")
else:
    print("Zip file not found. Please double-check the path in your Google Drive.")

Extraction complete!


### Explore and Load Extracted Files
Let's list the extracted files and display some data to verify it works.

In [3]:
# List extracted directories and files
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

/content/mimic_demo/mimic-iv-clinical-database-demo-2.2/demo_subject_id.csv


### 1. Load the Admissions Data and Inspect Columns

In [4]:
import pandas as pd
import numpy as np

# Load the admissions dataset
admissions_path = '/content/mimic_demo/mimic-iv-clinical-database-demo-2.2/hosp/admissions.csv.gz'
df_adm = pd.read_csv(admissions_path)

# Quick preview
display(df_adm.head())
print(df_adm.info())

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,NaN,URGENT,P03YMR,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicaid,ENGLISH,SINGLE,BLACK/CAPE VERDEAN,2196-02-24 12:15:00,2196-02-24 17:07:00,0
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,NaN,URGENT,P41R5N,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,?,MARRIED,HISPANIC/LATINO - PUERTO RICAN,NaN,NaN,0
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,NaN,URGENT,P233F6,TRANSFER FROM HOSPITAL,SKILLED NURSING FACILITY,Medicare,ENGLISH,MARRIED,WHITE,2134-08-17 16:24:00,2134-08-18 03:15:00,0
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,2111-11-15 17:20:00,URGENT,P38TI6,TRANSFER FROM HOSPITAL,DIED,Medicaid,ENGLISH,NaN,UNKNOWN,NaN,NaN,1
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,NaN,URGENT,P07HDB,TRANSFER FROM HOSPITAL,HOME,Other,ENGLISH,WIDOWED,WHITE,NaN,NaN,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   subject_id            275 non-null    int64 
 1   hadm_id               275 non-null    int64 
 2   admittime             275 non-null    object
 3   dischtime             275 non-null    object
 4   deathtime             15 non-null     object
 5   admission_type        275 non-null    object
 6   admit_provider_id     275 non-null    object
 7   admission_location    275 non-null    object
 8   discharge_location    233 non-null    object
 9   insurance             275 non-null    object
 10  language              275 non-null    object
 11  marital_status        263 non-null    object
 12  race                  275 non-null    object
 13  edregtime             182 non-null    object
 14  edouttime             182 non-null    object
 15  hospital_expire_flag  275 non-null    in

### 2. Preprocess and Exclude Hospital Deaths & Hospice Discharges

In [5]:
# Convert date columns to datetime objects
df_adm['admittime'] = pd.to_datetime(df_adm['admittime'])
df_adm['dischtime'] = pd.to_datetime(df_adm['dischtime'])

# Print unique discharge locations to find hospice categories
print("Discharge Locations:")
print(df_adm['discharge_location'].dropna().unique())

# Filter out hospital expirations (deaths) and hospice discharges
exclude_hospice_keywords = ['HOSPICE', 'Hospice']
hospice_mask = df_adm['discharge_location'].str.contains('|'.join(exclude_hospice_keywords), na=False, case=False)
death_mask = df_adm['hospital_expire_flag'] == 1

df_clean_adm = df_adm[~(death_mask | hospice_mask)].copy()

print(f"\nOriginal admissions count: {len(df_adm)}")
print(f"Admissions after excluding deaths and hospice: {len(df_clean_adm)}")

Discharge Locations:
['SKILLED NURSING FACILITY' 'HOME HEALTH CARE' 'DIED' 'HOME'
 'ACUTE HOSPITAL' 'CHRONIC/LONG TERM ACUTE CARE' 'REHAB' 'HOSPICE'
 'AGAINST ADVICE' 'PSYCH FACILITY']

Original admissions count: 275
Admissions after excluding deaths and hospice: 255


### 3. Compute 30-Day Readmission Metrics
We will sort the clean admissions by patient ID and admission time to identify consecutive stays.

In [6]:
# Sort chronologically per patient
df_clean_adm = df_clean_adm.sort_values(by=['subject_id', 'admittime'])

# Find the next admission time for each patient
df_clean_adm['next_admittime'] = df_clean_adm.groupby('subject_id')['admittime'].shift(-1)

# Compute days until next admission
df_clean_adm['days_to_next_admit'] = (df_clean_adm['next_admittime'] - df_clean_adm['dischtime']).dt.total_seconds() / (24 * 3600)

# Mark 30-day readmissions (days to next admission is positive and <= 30)
df_clean_adm['readmit_30'] = np.where((df_clean_adm['days_to_next_admit'] >= 0) & (df_clean_adm['days_to_next_admit'] <= 30), 1, 0)

# Calculate metrics
total_admissions = len(df_clean_adm)
patient_counts = df_clean_adm['subject_id'].value_counts()
patients_multi_adm = sum(patient_counts > 1)
total_readmissions_30 = df_clean_adm['readmit_30'].sum()

print(f"Total Clean Admissions: {total_admissions}")
print(f"Number of patients with more than one admission: {patients_multi_adm}")
print(f"Number of 30-day readmissions: {total_readmissions_30}")

Total Clean Admissions: 255
Number of patients with more than one admission: 41
Number of 30-day readmissions: 49


### Important Considerations & Contrast with UCI Diabetes Dataset

1. **Single-center boundary**: Because MIMIC is a database sourced from a single hospital system, we can only track readmissions *back to this specific facility*. If a patient is discharged and subsequently admitted to another hospital system within 30 days, we do not capture it.
2. **All-cause vs Disease-specific readmission**: Here, we measured readmissions across *all* diagnoses and causes. In contrast, datasets like the UCI Diabetes dataset explicitly focus on diabetes-related records and label readmissions specifically context-bound to clinical markers or specific cohorts (like diabetic patients under specific treatments).

### 4. Load the Diagnoses Data

In [7]:
# Load diagnoses dataset
diagnoses_path = '/content/mimic_demo/mimic-iv-clinical-database-demo-2.2/hosp/diagnoses_icd.csv.gz'
df_diag = pd.read_csv(diagnoses_path)

# Preview the unique values and formatting
display(df_diag.head())
print(df_diag.info())
print("ICD Versions present:", df_diag['icd_version'].unique())

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10035185,22580999,3,4139,9
1,10035185,22580999,10,V707,9
2,10035185,22580999,1,41401,9
3,10035185,22580999,9,3899,9
4,10035185,22580999,11,V8532,9


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4506 entries, 0 to 4505
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   subject_id   4506 non-null   int64 
 1   hadm_id      4506 non-null   int64 
 2   seq_num      4506 non-null   int64 
 3   icd_code     4506 non-null   object
 4   icd_version  4506 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 176.1+ KB
None
ICD Versions present: [ 9 10]


### 5. Define and Apply Diabetes Flags (ICD-9 & ICD-10)

In [8]:
# Strip any whitespace from the codes to ensure clean matches
df_diag['icd_code'] = df_diag['icd_code'].str.strip()

# ICD-9 definition: starts with '250'
is_icd9_diabetes = (df_diag['icd_version'] == 9) & (df_diag['icd_code'].str.startswith('250', na=False))

# ICD-10 definition: starts with E10, E11, E12, or E13
is_icd10_diabetes = (df_diag['icd_version'] == 10) & (df_diag['icd_code'].str.startswith(('E10', 'E11', 'E12', 'E13'), na=False))

# Flag diabetes records
df_diag['is_diabetes'] = is_icd9_diabetes | is_icd10_diabetes

# Identify unique patients with diabetes
diabetic_subjects = df_diag[df_diag['is_diabetes']]['subject_id'].unique()
total_demo_patients = df_diag['subject_id'].nunique()

print(f"Total unique patients in demo diagnoses: {total_demo_patients}")
print(f"Unique diabetic patients identified: {len(diabetic_subjects)}")
print(f"Percentage of patients with diabetes in the demo: {len(diabetic_subjects) / total_demo_patients * 100:.2f}%")

Total unique patients in demo diagnoses: 100
Unique diabetic patients identified: 35
Percentage of patients with diabetes in the demo: 35.00%


### Observations on Code-Harmonization

1. **Mixed ICD Schemes**: Since MIMIC-IV spans several years, it maps older stays using ICD-9 and newer stays using ICD-10. This requires explicit code translation or prefix mapping (as done above) to maintain consistency across the entire database.
2. **Granularity Differences**: ICD-10 codes are significantly more detailed and clinical than ICD-9 equivalents. Harmonization for the full dataset will require carefully aligning these hierarchical categories or mapping them to unified classification systems such as Clinical Classifications Software Refined (CCSR) or Elixhauser Comorbidity Indices.

### 6. Map UCI Features to MIMIC-IV and Compute Coverage in Demo

In [9]:
# Load necessary MIMIC-IV tables for feature extraction
base_path = '/content/mimic_demo/mimic-iv-clinical-database-demo-2.2'
patients_path = f"{base_path}/hosp/patients.csv.gz"
procedures_path = f"{base_path}/hosp/procedures_icd.csv.gz"
prescriptions_path = f"{base_path}/hosp/prescriptions.csv.gz"
services_path = f"{base_path}/hosp/services.csv.gz"
labevents_path = f"{base_path}/hosp/labevents.csv.gz"
d_labitems_path = f"{base_path}/hosp/d_labitems.csv.gz"

df_pat = pd.read_csv(patients_path)
df_proc = pd.read_csv(procedures_path)
df_pres = pd.read_csv(prescriptions_path)
df_serv = pd.read_csv(services_path)
df_labs = pd.read_csv(labevents_path)
df_dlabs = pd.read_csv(d_labitems_path)

In [12]:
# Let's align all features with the 'clean' admissions dataset (df_clean_adm)
# Merge admissions with patient demographics (age, gender)
df_feat = df_clean_adm.merge(df_pat[['subject_id', 'gender', 'anchor_age']], on='subject_id', how='left')

# Rename anchor_age to age to align with UCI name
df_feat['age'] = df_feat['anchor_age']

# Calculate time_in_hospital in days
df_feat['time_in_hospital'] = (df_feat['dischtime'] - df_feat['admittime']).dt.total_seconds() / (24 * 3600)

# Prior admission counts (number_inpatient, number_emergency)
df_feat = df_feat.sort_values(by=['subject_id', 'admittime'])
df_feat['number_inpatient'] = df_feat.groupby('subject_id').cumcount()  # cumulative count of prior admissions

# Prior emergency-type admissions (admission_type is 'URGENT' or 'EMERGENCY' in prior records)
df_feat['is_emerg'] = df_feat['admission_type'].str.contains('EMERGENCY|URGENT', case=False, na=False).astype(int)
df_feat['number_emergency'] = df_feat.groupby('subject_id')['is_emerg'].cumsum() - df_feat['is_emerg']

# Outpatient visits are not available in MIMIC clinical core (usually tracked in separate datasets or not recorded)
df_feat['number_outpatient'] = np.nan

# Count of labevents per admission
lab_counts = df_labs.groupby('hadm_id').size().rename('num_lab_procedures')
df_feat = df_feat.merge(lab_counts, on='hadm_id', how='left')

# Count of procedures per admission
proc_counts = df_proc.groupby('hadm_id').size().rename('num_procedures')
df_feat = df_feat.merge(proc_counts, on='hadm_id', how='left')

# Count of medications (distinct drugs in prescriptions)
med_counts = df_pres.groupby('hadm_id')['drug'].nunique().rename('num_medications')
df_feat = df_feat.merge(med_counts, on='hadm_id', how='left')

# Number of diagnoses
diag_counts = df_diag.groupby('hadm_id').size().rename('number_diagnoses')
df_feat = df_feat.merge(diag_counts, on='hadm_id', how='left')

# Principal diagnoses (diag_1, diag_2, diag_3 via seq_num)
for i in [1, 2, 3]:
    diag_sub = df_diag[df_diag['seq_num'] == i][['hadm_id', 'icd_code']].rename(columns={'icd_code': f'diag_{i}'})
    df_feat = df_feat.merge(diag_sub, on='hadm_id', how='left')

# Medical specialty (from services table: use the first service recorded for the admission)
df_first_service = df_serv.sort_values(by=['subject_id', 'transfertime']).groupby('hadm_id').first().reset_index()
df_feat = df_feat.merge(df_first_service[['hadm_id', 'curr_service']].rename(columns={'curr_service': 'medical_specialty'}), on='hadm_id', how='left')

# Payer code (mapping admissions.insurance directly)
df_feat['payer_code'] = df_feat['insurance']

# HbA1c & Glucose lab items mapping
# Let's search for itemids corresponding to HbA1c and glucose (using correct 'itemid' key)
hba1c_item_ids = df_dlabs[df_dlabs['label'].str.contains('HbA1c|Hemoglobin A1c', case=False, na=False)]['itemid'].tolist()
glucose_item_ids = df_dlabs[df_dlabs['label'].str.contains('Glucose', case=False, na=False)]['itemid'].tolist()

# Aggregate values using correct lab 'itemid' column
hba1c_events = df_labs[df_labs['itemid'].isin(hba1c_item_ids)].groupby('hadm_id')['valuenum'].max().rename('A1Cresult')
glucose_events = df_labs[df_labs['itemid'].isin(glucose_item_ids)].groupby('hadm_id')['valuenum'].max().rename('max_glu_serum')

df_feat = df_feat.merge(hba1c_events, on='hadm_id', how='left')
df_feat = df_feat.merge(glucose_events, on='hadm_id', how='left')

In [13]:
# Calculate coverage percentage for each target UCI feature in our final df_feat (aligned with clean admissions)
features_mapping = {
    'age': 'patients.anchor_age',
    'gender': 'patients.gender',
    'race': 'admissions.race',
    'time_in_hospital': 'dischtime - admittime',
    'number_inpatient': 'count of prior admissions (calculated)',
    'number_emergency': 'count of prior emergency admissions (calculated)',
    'number_outpatient': 'not available in MIMIC clinical core',
    'num_lab_procedures': 'count of labevents (calculated)',
    'num_procedures': 'count of procedures_icd (calculated)',
    'num_medications': 'distinct drugs in prescriptions (calculated)',
    'number_diagnoses': 'count of diagnoses_icd (calculated)',
    'diag_1': 'diagnoses_icd (seq_num 1)',
    'diag_2': 'diagnoses_icd (seq_num 2)',
    'diag_3': 'diagnoses_icd (seq_num 3)',
    'discharge_disposition': 'admissions.discharge_location',
    'admission_source': 'admissions.admission_location',
    'admission_type': 'admissions.admission_type',
    'A1Cresult': 'labevents (HbA1c items)',
    'max_glu_serum': 'labevents (Glucose items)',
    'payer_code': 'admissions.insurance',
    'medical_specialty': 'services.curr_service'
}

coverage_records = []
for uci_feat, mimic_src in features_mapping.items():
    if uci_feat in df_feat.columns:
        non_null_pct = df_feat[uci_feat].notnull().mean() * 100
    elif uci_feat == 'discharge_disposition':
        non_null_pct = df_feat['discharge_location'].notnull().mean() * 100
    elif uci_feat == 'admission_source':
        non_null_pct = df_feat['admission_location'].notnull().mean() * 100
    else:
        non_null_pct = 0.0

    coverage_records.append({
        'UCI Feature': uci_feat,
        'MIMIC Source': mimic_src,
        'Coverage in Demo (%)': f"{non_null_pct:.2f}%"
    })

df_coverage = pd.DataFrame(coverage_records)
display(df_coverage)

,UCI Feature,MIMIC Source,Coverage in Demo (%)
0,age,patients.anchor_age,100.00%
1,gender,patients.gender,100.00%
2,race,admissions.race,100.00%
3,time_in_hospital,dischtime - admittime,100.00%
4,number_inpatient,count of prior admissions (calculated),100.00%
5,number_emergency,count of prior emergency admissions (calculated),100.00%
6,number_outpatient,not available in MIMIC clinical core,0.00%
7,num_lab_procedures,count of labevents (calculated),90.98%
8,num_procedures,count of procedures_icd (calculated),65.88%
9,num_medications,distinct drugs in prescriptions (calculated),90.20%
